# 03｜単体幾何の検証と計測

任意の実数特徴量を1〜8次元で与えられます。画像入出力・表示はこの幾何APIに依存しません。
頂点の座標和、重心座標の和、元位置の復元を確認します。

実行時間はこの環境での観測値です。他手法との比較や他OSでの速度を示すものではありません。
配布時の実行ではテスト用C ABIを利用しているため、nanobindの呼出し費用は測定していません。

In [1]:
import os

# 検証実行時だけ明示的に選ぶ接続です。通常実行はnanobindを使います。
if os.environ.get("PERMUTOHEDRAL_TEST_BRIDGE") == "1":
    from scripts._project import activate_test_bridge
    activate_test_bridge()

import numpy as np
from PIL import Image
from IPython.display import display
from permutohedral_segmentation import LatticeConfig, MergeConfig, segment
from permutohedral_segmentation import _native
from permutohedral_segmentation.datasets import load_example
from permutohedral_segmentation.visualization import render_labels, overlay_boundaries

backend = getattr(_native, "__test_backend__", "nanobind")
print("実行した接続方式:", backend)


実行した接続方式: ctypes-test-adapter (nanobind not validated)


In [2]:
from permutohedral_segmentation import label_features, locate_simplices

checks = []
for dimension in range(1, 9):
    features = np.random.default_rng(2026 + dimension).uniform(-100, 100, (1000, dimension))
    vertices, weights, elevated = locate_simplices(features)
    reconstruction = np.einsum("ni,nij->nj", weights, vertices)
    error = float(np.max(np.abs(reconstruction - elevated)))
    np.testing.assert_array_equal(vertices.sum(axis=2), 0)
    np.testing.assert_allclose(weights.sum(axis=1), 1, atol=1e-12)
    assert weights.min() >= -1e-12
    assert error < 1e-9
    checks.append({"次元": dimension, "点数": len(features), "最大復元誤差": error})
checks

[{'次元': 1, '点数': 1000, '最大復元誤差': 0.0},
 {'次元': 2, '点数': 1000, '最大復元誤差': 5.684341886080802e-14},
 {'次元': 3, '点数': 1000, '最大復元誤差': 5.684341886080802e-14},
 {'次元': 4, '点数': 1000, '最大復元誤差': 1.1368683772161603e-13},
 {'次元': 5, '点数': 1000, '最大復元誤差': 1.1368683772161603e-13},
 {'次元': 6, '点数': 1000, '最大復元誤差': 2.2737367544323206e-13},
 {'次元': 7, '点数': 1000, '最大復元誤差': 2.2737367544323206e-13},
 {'次元': 8, '点数': 1000, '最大復元誤差': 2.2737367544323206e-13}]

In [3]:
from statistics import median
from time import perf_counter

image = load_example("peppers")
conditions = [
    ("xy", LatticeConfig("xy", 0.025, 0.01), None),
    ("xyrgb raw", LatticeConfig("xyrgb", 0.025, 0.01), None),
    ("xyrgb merged 300", LatticeConfig("xyrgb", 0.025, 0.01), MergeConfig(300)),
]
measurements = []
for name, lattice, merge in conditions:
    segment(image, lattice=lattice, merge=merge)  # 計測前の予備実行
    samples = []
    for _ in range(3):
        started = perf_counter()
        result = segment(image, lattice=lattice, merge=merge)
        samples.append((perf_counter() - started) * 1000)
    measurements.append({"条件": name, "領域数": result.region_count,
                         "中央値ms": median(samples), "各回ms": samples})
measurements

[{'条件': 'xy',
  '領域数': 405,
  '中央値ms': 15.269604999957664,
  '各回ms': [15.269604999957664, 14.639684000030684, 15.906164999933026]},
 {'条件': 'xyrgb raw',
  '領域数': 16350,
  '中央値ms': 59.58776600004967,
  '各回ms': [61.361284999975396, 59.58776600004967, 56.97635799992895]},
 {'条件': 'xyrgb merged 300',
  '領域数': 300,
  '中央値ms': 807.703421000042,
  '各回ms': [717.7534290001404, 935.6816720001007, 807.703421000042]}]

領域IDは入力行の出現順で付与します。格子境界付近の浮動小数点の挙動は、
異なるコンパイラ・演算環境で別途確認が必要です。測定値には特徴量生成・ラベル集計を含み、
画像読込・PNG書出し・表示を含みません。結果は `uv.lock` の代用ではありません。